In [ ]:
/**
 * Caching:
 * - Jo compute hum first time already kar chuke hai usse yaad kar lo, means
 *   kahi na kahi store kr lo. But isse hum kaha kaha store kr skte hai?
 *   > RAM
 *   > Hard Disk
 * - Aur agar ye data lost ho jae to kuch fark padega? No! because ye data regenerate
 *   kiya jaa sakta hai.
 * 
 * TradeOff:
 * - Hard Disk is slow, but persistent => We store variable data to compute
 * - RAM is faster, but temporary      => We store already computed data
 * - And this acts as a cache. 
 * 
 * So, ek database cache k tarah v to act kr skta hai? Let's say humein koi data prepare
 * krne m 20 tables ko join krna pad rha hai aur financial data m usually humein bahut
 * saare aggregate functions run karne hote hai, like SUM(), etc.
 * 
 * To find bank balance of any user, we have to traverse the full balance sheet, and
 * at the end, we will get our balance. But agar humein ye har baar karna pad rha hai
 * to it's an expensive operation. To kya iss computation ko cache kr skte hai? Yes!
 * So, caching basically means ki hum kisi pre-computed data ko store kr skte hai.
*/

In [ ]:
/**
 * Consistent Hashing:
 * - Mere paas ek database hai jisme yaa to hm kuch put krte hai ya fir kuch read krte
 *   hai, iske alawa hum aur kuch nhi kr skte hai.
 * - Kya iss DB ko hm caching k liye use kr skte hai? Yes! Redis DB.
 * - Redis is in-memory database
 * 
 * Remember:
 * - Database Replication = Making copy of DB
 * - CQRS = Denormalize and keep the data in different format in different DB
*/

In [ ]:
/**
 * Design a Distributed Cache [Interview Question]:
 * 
 * > Hum kisi company m kaam krte hai, waha par Postgres DB hai and unhone bola ki
 *   humara jo database hai uske upar bahut jyda READ & WRITE Operations ho rhe hai,
 *   to hum READ ko optimize krna chah rhe hai to make it faster.
 *   - Hum caching laga denge
 * 
 * > For caching, kon sa database use karein?
 *   - Redis Database
 * 
 * > What was your thought process behind choosing redis?
 *   - Humein key-value pair store krna hai, aur jab v retrieve krna ho to key denge
 *     aur humein value mil jaegi.
 *   - And jydatar industry m redis use hota hai isliye mujhe v sirf redis pta hai.
 * 
 * > Ab hum iss redis ko kisi server k upar deploy krnge.
 * 
 *     +--------+ -----------> +--------+
 *     |        |              |        |
 *     +--------+ <----------- +--------+
 *       Server                   Redis
 *        |  |                 +--------+
 *        |  +------->-------> |        |
 *        +--<-------<-------- |        |
 *                             +--------+
 *                                PG DB
 * 
 * > Humara jo main server hoga uske paas user ki request aaegi, fir wo redis m check
 *   karega ki kya mere paas cache m data available hai ya nahi. Agar available hai to
 *   redis db user data return kr dega, agar nahi hai to Postgres m jao, aur data ko
 *   query karo jisme bahut jyda time lagega.
 * > Once we get the data, usko cache kr k redis store m rakh lo, aur data ko return
 *   kar do.
 * 
 * Problem:
 * > Poori consistency kharab ho gyi hai. Likh PG DB m rhe hai, read Redis DB se
 *   kr rhe hai aur ek baar data redis m chala gya to kabhi jaega hi nhi,
 *   jaega.
 * 
 * Solution:
 * > Agar hum kisi chij pe cache laga rhe hai to uss data pe hm TTL (Time to Leave) laga
 *   skte hai.
 * > There are multiple ways to remove cache data from redis called Cache Eviction
 *   Policies:
 *   a. LRU (Least Recently Used): Removing the items that have not been accessed for
 *      the longest period of time.
 *   b. LFU (Least Frequently Used): Removes items that are accessed the least often
 *      assuming low-frequency items are less valuable.
 *   c. FIFO (First-In-First-Out): Evicts the oldest items added to the cache list,
 *      regardless of how often or recently they are accessed.
 *   d. Random: Randomly selects items to remove, which is fast but less efficient,
 *      sometimes used in specialized scenarios.
 * > Jab v hum kisi cache ko hit krnge to read the data to uske timestamp ko update kr
 *   denge. Uske baad agar humari storage full ho rhi hai to sabse least recently used
 *   cached data ko remove kr denge. This is called LRU Cache.
 *   (Note: Design LRU Cache (LLD) [Interview Question])
 * > Isme har cache data ko head se hata k tail k aur move krte jaate hai, and it's a
 *   doubly linked list because we have to move back and forth. We're basically trading
 *   off storage for speed. And this concept is called READ THROUGHT CACHE.
 * > Isme agar data redis ko mila to theek warna DB se query kr k lao, cache pe update
 *   kr do aur wapas aa jao.
 * 
 * Problem:
 * > Agar ek saath 1000 users aa gya aur same data dhund rhe hai cache m, agar data nhi
 *   mila to multiple users will hit the database for the same data.
 * 
 * Solution: LOCKS
 * > Hum redis k andar ek flag laga denge, jo bataega ki iska cache abhi nhi hai but
 *   I am going to prepare, taaki agar thousand requests aaengi ki data abhi hai nhi
 *   lekin queued hai, to wait krnge, aur jisne ye lock acquire kiya wo jaa k data prepare
 *   karega.
 * 
 * Problem:
 * > Let's assume mere paas ek redis ka bahut bada server hai jo 1 billion operations 
 *   handle kar sakta hai. Ek point pe saara ka saara data cache ho jaega. Mere postgres
 *   k upar load hai nahi, ab saara load redis pe chala gya. Now redis becomes the 
 *   bottleneck because everyone is coming to our redis which now acts as Primary DB. 
 *   TTL will remove cache after 24hrs, but what if 1 billion cache within 24 hrs m ho
 *    jae. How to scale this redis server?
 * 
 * Solution: Sharding
 * > Either we can horizontally scale or vertically. Agar hum isse vertically scale
 *   karnge to downtime aaega, aur single point of failure v banega. 
 * > Humne isko horizontally scale kra aur multiple partitions (3 shards) bana diya.
 * > p0, p1, and p2 are three independent servers, jo ek tarah ka distributed cache
 *   ban gya, aur humne ek hash function likha jiske andar apna key ko daalnge:
 *   => hashFn(key) = key % lengthOfServer => key % 3 = ?
 * > Agar iska answer 0 aaya to data ko p0 k rakhunga, 1 aaya p1, or 2 aaya to p2.
 *   Isse kabhi v hotspot wali problem nahi aaegi because humne equally 3 m divided kr
 *   diya data ko, jisse kabhi v humara ek server down nhi jaega because now it is
 *   split into 3 different servers.
 * 
 * Problem: Adding or removing DB Partitions
 * > Jab hm apne servers increase krnge to p3 partition add krna pdega, but ek aur
 *   server add krte hi mera hash function badal gya:
 *   => hashFn(key) = key % lengthOfServer => key % 4 = ?
 * > Another problem is ek baar hashfunction change ho gya to jise partition pe humne
 *   data store kiya tha jab read krnge to humein invalid data milega kyuki hum dusre
 *   partition pe check kr rhe honge.
 * > Everytime a new server is added or removed, we have to move the data:
 *   (length of server = 3 -> 4)
 *   0  0  0  -> No movement of data 
 *   1  1  1  -> No movement of data
 *   2  2  2  -> No movement of data
 *   3  0  3  -> Move data from 0 to 3
 *   4  1  0  -> Move data from 1 to 0
 *   5  2  1  -> Move data from 2 to 1
 *   6  0  2  -> Move data from 0 to 2
 *   7  1  3  -> Move data from 1 to 3
 *   8  2  0  -> Move data from 2 to 0
 * > Sahi to sirf upar k 3 rahe baaki sb data humein move krne pada sirf ek server
 *   increase krne par. What if ye 1M entries hoti to? Data Movement is not good thing.
 * > Kya humara hash function consistent tha? No!, server down jaega to hash function
 *   badlega, fir server up jaega to fir se hash function badlega. So, our hash function
 *   is not consistent.
 * 
 * Solution: Consistent Hashing
 * > Let's try to make our hash function consistent which will reduce the data movement.
 * > Hum ek Hash Function likhnge jo humein humesa ek consistent value dega. Let's say
 *   humne apna Hash Function likha: Hash = key % 1000 jo kabhi change nhi hoga. But kya
 *   mai kabhi 1000 server lagaunga? No!
 * > Hum consistent hashing m ek ring banate hai, and assume karo isme 1000 tak numbers
 *   hai like clock, aur iss ring ko hum kabhi v change nhi krnge. To kya isse meri
 *   hash function consistent hui? Yes! 
 * > Ab pehla din humne 3 server liya, aur inn teeno ko humein ring pe place krna hai.
 *   But isse kaha aur kaise place krnge? Inn servers ka kuch public IP hoga:
 *   a. 10.2.3.4
 *   b. 11.4.6.7
 *   c. 30.2.1.2
 * > Inn Public IP ko hm apne hash function m daal denge, and suppose hash id = 712
 *   aaya aur waha pe hum isse place kr denge, similarly, baaki servers k v saath wahi
 *   krnge i.e., random 3 places pe 3 servers place kr diya.
 * > Ab jab v humare paas koi data aaega to humein uss key-value pair data ko hash
 *   function m daalna hai, aur humein randomly ring k upar koi space milega like 655.
 *   But 655 pe koi server nhi hai, to humein yha pe clockwise move krna hai aur server
 *   dhundna hai, aur fir 712 wale server pe ye data store kr dega, and similalry jo
 *   v data aaega usko v aise hi store krnge.
 * > Hum apna hash function kabhi nhi badlega, pehle data to place karo fir clockwise
 *   direction pe server dhundna start karo aur usme data store kr do.
 * 
 * Problem: Adding or removing servers
 * 
 * Solution:
 * > Ab hum ek aur server le k aae jisko uski position 670 pe mili.
 * > Mathematically, pichle server se le k iss server tak bas humein ye data move krna
 *   padega i.e., move data 655 from 712 server to 670 server. Basically, pichle server
 *   se pehle se le kr next server tak k bich m jitna data tha sabko server 670 m move
 *   kr dena hai
 * > Ab isse hoga ye ki chahe hm server add kre ya remove, humara hash function 
 *   consistent rahega and data movement minimal rahega. And this is called 
 *   Consistent Hashing
*/